In [51]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import  TypedDict
from dotenv import load_dotenv

In [52]:
load_dotenv()

True

In [53]:
model=ChatGroq(model='llama-3.1-8b-instant', temperature=0.4, max_retries=2)

In [54]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    score:float


In [55]:
def create_outline(state:BlogState)->BlogState:
    title=state['title']
    prompt=f'Generate a detailed outline for the topic {title}'
    outline=model.invoke(prompt).content
    state['outline']=outline
    return state

In [56]:
def create_blog(state:BlogState)->BlogState:
    title=state['title']
    outline= state['outline']
    prompt= f'Write a blog on the title: {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content
    state['content']=content
    return state

In [57]:
def rate_blog(state:BlogState)->BlogState:
    outline=state['outline']
    content=state['content']
    prompt=f'Based on this outline {outline}, rate this blog {content} out of 10. \n Give me output with just one value which should be valid literal for int()'
    score=model.invoke(prompt).content
    state['score']=int(score)
    return state

In [58]:
graph= StateGraph(BlogState)
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('rate_blog', rate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'rate_blog')
graph.add_edge('rate_blog', END)


workflow=graph.compile()

In [59]:
initial_state={'title': 'Deep Learning in computer vision'}
final_state= workflow.invoke(initial_state)
print(final_state['score'])

8
